# Chapter 3: Enabling actions: Tool use

In [1]:
import json
from dotenv import load_dotenv
from litellm import completion
load_dotenv()

True

### Listing 3.1: Calculator Tool Definition Schema

In [2]:
calculator_tool_definition = {
    "type": "function",
    "function": {
        "name": "calculator",
        "description": "Perform basic arithmetic operations.",
        "parameters": {
            "type": "object",
            "properties": {
                "operator": {
                    "type": "string",
                    "description": "Arithmetic operation to perform",
                    "enum": ["add", "subtract", "multiply", "divide"]
                },
                "first_number": {
                    "type": "number",
                    "description": "First number for the calculation"
                },
                "second_number": {
                    "type": "number",
                    "description": "Second number for the calculation"
                }
            },
            "required": ["operator", "first_number", "second_number"]
        }
    }
}          

### Listing 3.2: Implementation of Calculator Function

In [3]:
def calculator(operator: str, first_number: float, second_number: float):
    if operator == "add":
        return first_number + second_number
    elif operator == "subtract":
        return first_number - second_number
    elif operator == "multiply":
        return first_number * second_number
    elif operator == "divide":
        if second_number == 0:
            raise ValueError("Cannot divide by zero")
        return first_number / second_number
    else:
        raise ValueError(f"Unsupported operator: {operator}")

### Listing 3.3: Tool-calling execution example

In [4]:
tools = [calculator_tool_definition]

response_without_tool = completion(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": "What is the capital of South Korea?"}],
    tools=tools
)

print(response_without_tool.choices[0].message.content)
print(response_without_tool.choices[0].message.tool_calls)

response_with_tool = completion(
        model='gpt-5-mini',
        messages=[{"role": "user", "content": "What is 1234 x 5678?"}],
        tools=tools
)
print(response_with_tool.choices[0].message.content)
print(response_with_tool.choices[0].message.tool_calls)

The capital of South Korea is Seoul (Korean: 서울).
None
None
[ChatCompletionMessageToolCall(function=Function(arguments='{"operator":"multiply","first_number":1234,"second_number":5678}', name='calculator'), id='call_wxD6YZTXYgdpME1sZWLJKyUM', type='function')]


### Listing 3.4: Extracting and executing the tool from the response

In [5]:
ai_message = response_with_tool.choices[0].message

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

if function_name == "calculator":
    result = calculator(**function_args)

In [6]:
print(type(tool_call.function.arguments), type(function_args), result)

<class 'str'> <class 'dict'> 7006652


### Listing 3.5: Feeding tool results back to the LLM

In [9]:
messages=[{"role": "user", "content": "What is 1234 x 5678?"}]
messages.append({"role": "assistant", "content": ai_message.content, "tool_calls": ai_message.tool_calls})

if ai_message.tool_calls:
    for tool_call in ai_message.tool_calls:
        function_name = tool_call.function.name
        function_args = json.loads(tool_call.function.arguments)

        if function_name == "calculator":
            result = calculator(**function_args)

            messages.append({"role": "tool", "content": str(result), "tool_call_id": tool_call.id})

final_response = completion(
    model="gpt-5-mini",
    messages=messages
)
print("Messages:", messages)
print("Final_answer:", final_response.choices[0].message.content)

Messages: [{'role': 'user', 'content': 'What is 1234 x 5678?'}, {'role': 'assistant', 'content': None, 'tool_calls': [ChatCompletionMessageToolCall(function=Function(arguments='{"operator":"multiply","first_number":1234,"second_number":5678}', name='calculator'), id='call_wxD6YZTXYgdpME1sZWLJKyUM', type='function')]}, {'role': 'tool', 'content': '7006652', 'tool_call_id': 'call_wxD6YZTXYgdpME1sZWLJKyUM'}]


AttributeError: 'Choices' object has no attribute 'messages'

In [10]:
print("Final answer:", final_response.choices[0].message.content)

Final answer: 1234 × 5678 = 7,006,652
